# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Content Refresh Prioritization** is a **binary classification** problem: predict whether a page is declining (`is_declining_label = 1`) or not (`0`). The output gets used as a ranking signal — pages with higher predicted probability of declining float to the top of the review queue.

It's not a pure ranking task because I don't have pairwise preferences, and it's not clustering because the label is already defined. Classification with probability scores felt like the cleanest framing for this.

In [3]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ShamirAli55/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

## 2. Target or proxy

The prediction target is **`is_declining_label`**, derived from `trend_direction == 'down'`.

This is a **rule-based proxy**, not a measured outcome. We're defining decline as "the trend went down" — which is observable but one step removed from causality. That's fine for prioritization, as long as we don't claim the model is doing anything beyond ranking pages by their similarity to the declining pattern in this dataset.

Note: `trend_direction` and `trend_pct` are **excluded** from features — they directly encode the label.

In [4]:
df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

**Precision@50** — out of the top 50 pages the model flags, what fraction are genuinely declining?

Why not accuracy? Because 54% of pages are declining anyway, a model that flags everything gets 0.54 accuracy for free. That's not useful.

Why not recall? Because the reviewers can only look at a limited number of pages. Getting the right ones into the top 50 matters more than finding every declining page across the whole dataset.

The baseline (scripts/02_baseline_score.py) achieves Precision@50 = 0.240. That's the number to beat.

In [5]:
print("Declining pages:", df["is_declining_label"].sum())
print("Non-declining pages:", len(df) - df["is_declining_label"].sum())
naive_rate = df["is_declining_label"].mean()
print(f"Naive baseline (flag everything): Precision@50 would reflect dataset rate of {naive_rate*100:.1f}%")

Declining pages: 16262
Non-declining pages: 13738
Naive baseline (flag everything): Precision@50 would reflect dataset rate of 54.2%


## 4. The unit of analysis, as a real dataframe

**One row = one content page.** Each row has a snapshot of that page's search performance at a point in time — impressions, CTR, average position, content age, etc. The label (`is_declining_label`) says whether that page was on a downward trend.

The model will predict one probability per row, and we sort descending to get the refresh queue.

In [6]:
display(
    df[
        [
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "trend_direction",
            "is_declining_label"
        ]
    ].head()
)

print("Total Pages:", len(df))

Total Pages: 30000


## 5. Why ML beats a fixed rule here

A fixed rule like "flag pages older than 180 days" or "flag pages with impressions > 10,000" only uses one signal at a time. In the data, declining pages come in different shapes — some are old with decent CTR, some are newer with collapsed impressions, some have high average position but zero clicks.

A model can consider all these signals simultaneously and adjust to their interactions. The reference pipeline already shows a ~3x lift over a simple hand rule using Precision@50 — that gap is the reason ML is worth trying here rather than just sorting by one column.

In [7]:
print("Average CTR:", round(df["ctr"].mean(), 4))
print("Average Content Age:", round(df["content_age_days"].mean(), 2))
print("Average Position:", round(df["avg_position"].mean(), 2))

Average CTR: 0.5107
Average Content Age: 256.17
Average Position: 16.34


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.